# Reduction_dataset_V2

Deuxieme version de la réduction du dataset --> **celle qui doit être utilisée**

## **Objectif** : 
Ne conserver que le TOP **12** des articles qui réprésentent **75%** des ventes totales

## **Resultat** : 
Liste des articles conservés (top 12 articles) :

[**'ECLAIR', 'COOKIE', 'CAMPAGNE', 'BOULE 400G', 'TARTELETTE', 'SPECIAL BREAD', 'CEREAL BAGUETTE', 'BAGUETTE', 'BANETTE', 'PAIN AU CHOCOLAT', 'CROISSANT', 'TRADITIONAL BAGUETTE']**

# 1. Import du dataset original 

In [103]:
# importer le dataset
import pandas as pd
data = pd.read_csv('data_tmp/bakery_sales.csv')
data.head()

print(f"Nombre de lignes avant modifications = {len(data)}")
print(f"Nombre d'article unique avant modifications = {data['article'].nunique()}")

Nombre de lignes avant modifications = 234005
Nombre d'article unique avant modifications = 149


# 2. Exploration et nettoyage  du dataset. 

## Résultat

- Aucune ligne avec des informations manquantes ou abérrantes
- 1291 remboursements. Pour chaque 2 lignes : un ligne d'achat et une ligne de remboursement;  donc 2582 lignes supprimées 
- Suppression des articles "coupe" et "formule sandwich" parmit les tops articles
- Tous les regroupements ou suppression effectués dans la V1 ne sont pas mis en application dans cette V2. En effet dans cette V2 on se contente de ne garder que les **TOP 12** articles

## A. Correction des quantités négatives

Il y avait certaines lignes avec des quantités négatives. Après études ça corrrespond à des articles qui ont été remboursés. Nous supprimons donc les lignes négatives et les lignes des achats correspondants

In [104]:
# Nombre de lignes avant modifications
nb_avant =  len(data)
print(f"Nombre de lignes avant modifications = {len(data)}")

### On commence par récuperer les ticket_number de tous les tickets négatifs
remb_tickets = data.loc[data['Quantity'] < 0, 'ticket_number']

### On récupère les ticket_number des tickets qu'ils remboursent et on les join dans un set
tickets_a_supprimer = set(remb_tickets).union(remb_tickets - 1)

### Enfin on supprimme de la BDD tous les tickets number des tickets à rembourser et les tickets remboursés
data = data[~data['ticket_number'].isin(tickets_a_supprimer)]

# Nombre de lignes après modifications
nb_apres = len(data)
print(f"Nombre de lignes supprimées = {nb_avant - nb_apres}")
print(f"Nombre de lignes après modifications = {len(data)}")


Nombre de lignes avant modifications = 234005
Nombre de lignes supprimées = 2582
Nombre de lignes après modifications = 231423


## B. suppression de l'article "coupe" et "formule sandwich"

L'article "coupe" correspond à une prestation et non à un poduit

In [105]:
#Supprimer artcle 'coupe'
data = data[data['article'] != 'COUPE']
data = data[data['article'] != 'FORMULE SANDWICH']


# 3. Reduction du dataset : on ne conserve que le TOP **12** articles 

**Résultats avant réduction** :

- Nombre d'article unique avant Reduction = 146
- Nombre de lignes avant Reduction : 207068
- Nombre de d'article vendues avant Reduction : 331332

**Résultats après réduction** :

- Nombre d'article unique après Reduction = 12
- Nombre de lignes après Reduction : 143702
- Nombre de d'article vendues après Reduction : 251675

In [106]:
print(f"Nombre d'article unique avant Suppression = {data['article'].nunique()}")
print(f"Nombre de lignes avant suppression des articles très peu vendus : {len(data)}")
ventes_par_article = data.groupby("article")["Quantity"].sum()
ventes_totales = ventes_par_article.sum()
print(f"Nombre de d'article vendues avant suppression des articles très peu vendus : {ventes_totales}")



# Définir le seuil en pourcentage des ventes totales
seuil = 0.25
# Connaitre les articles dont le cumul représente moins que le seuil
seuil_pct_ventes = ventes_totales * seuil 


# Trier les articles du moins vendu au plus vendu (par unités vendues)
ventes_tries = ventes_par_article.sort_values(ascending=True)
# Somme cumulée des ventes en partant du moins vendu
cumul_ventes = ventes_tries.cumsum()
# Articles dont le cumul des ventes dépasse le seuil en pourcentage des ventes totales
articles_sous_pct = ventes_tries[cumul_ventes <= seuil_pct_ventes]
top_articles = ventes_tries[cumul_ventes > seuil_pct_ventes]

print(f"En conservant les top articles qui représentent plus de {100 - seuil*100}% des ventes, on conserve {len(top_articles)} articles sur {len(ventes_par_article)} articles au total.")

# suppression des lignes avec des articles très peu vendus
articles_a_supprimer = articles_sous_pct.index.tolist()
data = data[~data["article"].isin(articles_a_supprimer)]
print(f"Nombre d'article unique après Suppression = {data['article'].nunique()}")
print(f"Nombre de lignes après suppression des articles très peu vendus : {len(data)}")
print(f"Nombre de d'article vendues après suppression des articles très peu vendus : {data['Quantity'].sum()}")




Nombre d'article unique avant Suppression = 146
Nombre de lignes avant suppression des articles très peu vendus : 207068
Nombre de d'article vendues avant suppression des articles très peu vendus : 331332.0
En conservant les top articles qui représentent plus de 75.0% des ventes, on conserve 12 articles sur 146 articles au total.
Nombre d'article unique après Suppression = 12
Nombre de lignes après suppression des articles très peu vendus : 143702
Nombre de d'article vendues après suppression des articles très peu vendus : 251675.0


# 4. Data to csv 

In [107]:
data.to_csv('data_tmp/bakery_sales_top12.csv', index=False)